# Explore the data, the task and your predictions1. the tables2. one sample, one figure — the tool you will use most3. the visualisation toolkit: pictures, series, maps, counts4. what the data loader hands the model5. the predictions of a training runRun it from the repository folder with `DISFOR_DATA_ROOT` set. Nothing here needs a GPU.

In [ ]:
import os
from pathlib import Path

import polars as pl

from forest_disturbance.build import build_datamodule, build_mapping
from forest_disturbance.config import load_config
from forest_disturbance.metrics.events import s2_dates
from forest_disturbance.metrics.non_operational import evaluate_non_operational
from forest_disturbance.metrics.operational import evaluate_operational
from forest_disturbance.viz import (
    Sample,
    plot_class_counts,
    plot_code_counts,
    plot_hexmap,
    plot_hexmap_grid,
    plot_sample,
    style,
)

os.chdir(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
config_file = next(path for path in sorted(Path("configs").glob("*.yaml")) if "mapping" not in path.name)
config = load_config(config_file)
root = Path(config["data"]["root"])
mapping = build_mapping(config)
style.use()  # the colours, fonts and sizes every figure in the docs uses
print("config:", config_file, "| data:", root)

## 1. The tables

In [ ]:
samples = pl.read_parquet(root / "samples.parquet")
labels = pl.read_parquet(root / "labels.parquet")
splits = pl.read_parquet(root / "splits.parquet")
frames = pl.read_parquet(root / "zarr_frames.parquet")
print(samples.height, "samples,", labels.height, "label rows,", frames.height, "usable images")
labels.head(10)

In [ ]:
# Every raw label code, and the class the benchmark turns it into.
figure = plot_code_counts(labels, mapping)

In [ ]:
# The six disturbance classes, split by the fold that validates them.
figure = plot_class_counts(labels, mapping, splits)

## 2. One sample, one figure`plot_sample` is the function you will use most. Every panel is optional:| argument | what it adds ||---|---|| `patches=("s2", "s1", "index", "pca")` | strips of pictures (needs the zarr patches) || `series=("NDVI", ("VV", "VH"))` | one panel per entry; a tuple shares one panel || `predictions=` | the model's class probabilities || `targets=` | what the loss asks for on every date || `crop=`, `n_patches=`, `span=` | how much ground, how many dates, which period |Names you can put in `series`: the 12 Sentinel-2 bands, `VV` and `VH`,`NDVI NDMI NDWI NBR NDRE`, and `PC1 PC2 PC3` (principal components of all the bands).

In [ ]:
sample_id = 889
figure = plot_sample(sample_id, root, series=("NDVI", ("VV", "VH")))

In [ ]:
# With pictures. Drop `patches` if you do not have the zarr patches on this machine.
figure = plot_sample(
    sample_id, root, patches=("s2", "s1"), series=("NDVI",), n_patches=5, crop=96
)
style.save(figure, "plots/one_sample.png")  # 200 dpi, tight margins, ready for a report

In [ ]:
# Pick a random sample with a given disturbance class.
wind_codes = [code for code, c in mapping.code_to_class.items() if mapping.class_names[c] == "Wind"]
candidates = labels.filter(pl.col("label").is_in(wind_codes))["sample_id"].unique()
figure = plot_sample(int(candidates.sample(1, seed=1)[0]), root)

## 3. The visualisation toolkitSame function, different questions. Try changing a band triplet or a series name andlook again — this is the cheapest way to get a feel for what a disturbance looks like.

In [ ]:
# False colour: healthy vegetation is bright red, bare ground is blue-green.
figure = plot_sample(
    sample_id, root, patches=("s2",), series=(), annotations=False,
    s2_bands=("B08", "B04", "B03"), crop=96,
)

In [ ]:
# Three indices as one picture, and a PCA of all 12 bands.
figure = plot_sample(sample_id, root, patches=("index", "pca"), series=(), annotations=False)

In [ ]:
# Several series on one panel (z-scored when the units differ), or one panel each.
figure = plot_sample(sample_id, root, series=(("NDVI", "NDMI", "NDWI"), ("PC1", "PC2")))

In [ ]:
# The numbers behind the panels, if you would rather compute with them.
sample = Sample.load(sample_id, root)
dates, ndvi = sample.series("NDVI")
print(sample.caption())
print(len(dates), "Sentinel-2 dates, NDVI from", round(float(ndvi.min()), 2), "to", round(float(ndvi.max()), 2))
sample.events()

In [ ]:
# Where the samples are. `cell_km` sets the size of a hexagon.
figure, ax = plot_hexmap(samples, log_scale=True, cell_km=80, title=f"{samples.height} sampled pixels")

In [ ]:
# One map per class: this is what makes the sampling design visible.
groups = {}
for class_id in sorted(set(mapping.code_to_class.values())):
    codes = [c for c, k in mapping.code_to_class.items() if k == class_id]
    ids = labels.filter(pl.col("label").is_in(codes)).select("sample_id").unique()
    groups[mapping.class_names[class_id]] = samples.join(ids, on="sample_id")
figure = plot_hexmap_grid(groups, ncols=3, cell_km=150, chips=True)

## 4. What the data loader hands the modelOne example = the recent and the yearly series around a target date, plus the target class.

In [ ]:
datamodule = build_datamodule(config, mapping)
datamodule.setup()
dataset = datamodule.val_set
print(len(datamodule.train_set), "training examples,", len(dataset), "validation examples")
dataset.examples.head()

In [ ]:
example = dataset[1000]
print("sample", example["sample_id"], "target date", example["target_date"], "target", example["target"])
for name, series in [
    ("recent", example["inputs"]["s2"]["recent"]),
    ("yearly", example["inputs"]["s2"]["yearly"][0]),
]:
    print(name, series.dates, tuple(series.values.shape))

In [ ]:
batch = next(iter(datamodule.val_dataloader()))
{key: tuple(value.shape) for key, value in batch["inputs"]["s2"]["recent"].items()}

In [ ]:
# The same thing as a picture: the target class of every date, and how old the event is.
figure = plot_sample(
    int(dataset.examples["sample_id"][0]),
    root,
    series=("NDVI",),
    targets=dataset.examples,
    forget_days=config["loss"]["forget_days"],
)

## 5. Predictions of a training runPoint `run` at a folder in `runs/`. One probability column per class, per date.

In [ ]:
run = sorted(Path("runs").glob("*"))[-1]  # the most recent run
prediction_file = sorted((run / "predictions").glob("epoch_*[0-9].parquet"))[-1]
predictions = pl.read_parquet(prediction_file)
print(prediction_file)
predictions.head()

In [ ]:
no = evaluate_non_operational(predictions, labels=labels, s2_dates=s2_dates(frames), mapping=mapping)
o = evaluate_operational(
    predictions, labels=labels, timeline=frames, mapping=mapping,
    **config["evaluation"]["operational"],
)
pl.DataFrame(
    [{"metric": k, "value": round(v, 3)} for k, v in no.items() if k.endswith("60d") and v is not None]
    + [{"metric": f"operational binary {k}", "value": round(v, 3)} for k, v in o["binary"].items()]
)

In [ ]:
# A validation sample with an event: the model's probabilities and the alerts it would send.
events = labels.filter(
    pl.col("sample_id").is_in(predictions["sample_id"].unique().implode())
    & pl.col("label").is_in(list(mapping.code_to_class))
)
figure = plot_sample(
    int(events["sample_id"][0]), root, series=("NDVI",),
    predictions=predictions, alerts=o["alerts"],
)